# 02 - Prong 1: PyG ID-GNN Training

Structural embeddings on the heterogeneous consumption graph.


In [ ]:
import torch
from bgi_trident.graph.pyg.model import PyGProng


In [ ]:
# Load graph from notebook 01
from bgi_trident.graph.builder import ConsumptionGraphBuilder, InteractionData
import pandas as pd
from pathlib import Path
DATA = Path('../src/data')
data = InteractionData(food_orders=pd.read_csv(DATA/'interactions/food_orders.csv'), instamart_orders=pd.read_csv(DATA/'interactions/instamart_orders.csv'), dineout_bookings=pd.read_csv(DATA/'interactions/dineout_bookings.csv'), users=pd.read_csv(DATA/'bangalore_users.csv'), restaurants=pd.read_csv(DATA/'bangalore_restaurants.csv'), products=pd.read_csv(DATA/'bangalore_products.csv'), venues=pd.read_csv(DATA/'bangalore_venues.csv'))
builder = ConsumptionGraphBuilder(data).build()
pyg_data = builder.to_pyg()


In [ ]:
node_dims = {nt: pyg_data[nt].x.size(1) for nt in pyg_data.node_types if hasattr(pyg_data[nt], 'x')}
model = PyGProng(node_feature_dims=node_dims, hidden_dim=128, embed_dim=128, num_layers=2)
print(f'Parameters: {sum(p.numel() for p in model.parameters()):,}')


In [ ]:
with torch.no_grad():
    embeddings = model.encode(pyg_data)
for ntype, emb in embeddings.items():
    print(f'{ntype}: {emb.shape}')
